In [1]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
%cd ~/multi-modal/Proposal2
from huggingface_hub import snapshot_download
path = snapshot_download("tahoebio/Rhaister", local_dir="Rhaister")
print("downloaded to", path)
%pip install -q -e ./Rhaister
!ls Rhaister Rhaister/rhaister Rhaister/splits Rhaister/splits/parse Rhaister/scripts

/home/w202gxg/multi-modal/Proposal2
downloaded to /home/w202gxg/multi-modal/Proposal2/Rhaister

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Rhaister:
autoresearch  figures	      README.md			 scripts    uv.lock
CLAUDE.md     LICENSE	      results_sensitivity.jsonl  splits
config.json   plots	      rhaister			 tests
docs	      pyproject.toml  rhaister.egg-info		 tutorials

Rhaister/rhaister:
eval_splits.py	      prepare.py	      train_sensitivity.py
__init__.py	      prepare_sensitivity.py  train_zeroshot.py
prepare_cell_eval.py  state_metrics.py
prepare_combined.py   train.py

Rhaister/scripts:
baseline_sensitivity.py  run_parse_sweep.py
baselines.py		 run_replogle_nadig_sweep.py
data_prep		 run_titrations_cells.py
eval_a_vs_b.py		 run_titrations_cells_zeroshot.py
eval_permute.py		 run_titrations_drugs.py
fetch_state.py		 zeroshot_baselines.py
fetch_wandb

In [3]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
%cd ~/multi-modal/Proposal2
from huggingface_hub import HfApi, snapshot_download

# list files and sizes without downloading
info = HfApi().model_info("arcinstitute/ST-SE-Parse", files_metadata=True)
for f in info.siblings:
    print(f"{(f.size or 0) / 1e6:10.1f} MB  {f.rfilename}")

# download only small config / metadata files
snapshot_download("arcinstitute/ST-SE-Parse", local_dir="ST-SE-Parse",
                  allow_patterns=["*.yaml", "*.yml", "*.toml", "*.json", "*.md", "*.txt"])
!find ST-SE-Parse -type f | head -30

/home/w202gxg/multi-modal/Proposal2
       0.0 MB  .gitattributes
       0.0 MB  MODEL_ACCEPTABLE_USE_POLICY.md
       0.0 MB  MODEL_LICENSE.md
       0.0 MB  README.md
       0.0 MB  fewshot/split_0.toml
       0.0 MB  fewshot/split_0/batch_onehot_map.pkl
       0.0 MB  fewshot/split_0/cell_type_onehot_map.pkl
     578.6 MB  fewshot/split_0/checkpoints/best.ckpt
     578.6 MB  fewshot/split_0/checkpoints/final.ckpt
     578.6 MB  fewshot/split_0/checkpoints/last.ckpt
       0.0 MB  fewshot/split_0/config.yaml
       0.0 MB  fewshot/split_0/data_module.torch
       0.0 MB  fewshot/split_0/eval_best.ckpt/B_Intermediate_Memory_agg_results.csv
      15.1 MB  fewshot/split_0/eval_best.ckpt/B_Intermediate_Memory_pred_de.csv
      15.6 MB  fewshot/split_0/eval_best.ckpt/B_Intermediate_Memory_real_de.csv
       0.0 MB  fewshot/split_0/eval_best.ckpt/B_Intermediate_Memory_results.csv
    3846.9 MB  fewshot/split_0/eval_best.ckpt/adata_pred.h5ad
    3846.9 MB  fewshot/split_0/eval_best.ckpt/ada

In [4]:
import glob, yaml
for f in sorted(glob.glob("ST-SE-Parse/fewshot/split_*.toml")):
    print("=====", f); print(open(f).read()[:1500])

cfg = yaml.safe_load(open("ST-SE-Parse/fewshot/split_0/config.yaml"))
print("===== config data section")
print(yaml.dump(cfg.get("data", cfg), default_flow_style=False)[:2000])

===== ST-SE-Parse/fewshot/split_0.toml
# Dataset paths - maps dataset names to their directories
[datasets]
parse = "/data/parse_final_concat"

# Training specifications
# All cell types in a dataset automatically go into training (excluding zeroshot/fewshot overrides)
[training]
parse = "train"

# Zeroshot specifications - entire cell types go to val or test
[zeroshot]

# Fewshot specifications - explicit perturbation lists
[fewshot]
[fewshot."parse.B_Intermediate_Memory"]
val = ['G-CSF', 'IFN-beta', 'M-CSF', 'IGF-1']
test = ['C5a', 'TWEAK', 'LIF', 'IL-17C', 'FGF-beta', 'BAFF', 'FLT3L', 'IFN-lambda2', '4-1BBL', 'IL-17E', 'IL-2', 'IL-19', 'PRL', 'IL-16', 'IL-7', 'GM-CSF', 'IL-17A', 'IL-13', 'IL-36-alpha', 'PSPN', 'EPO', 'IL-9', 'IL-31', 'IL-11', 'LT-alpha1-beta2', 'TL1A', 'IL-32-beta', 'IFN-omega', 'IL-8', 'Noggin', 'IFN-epsilon', 'IL-17F', 'Leptin', 'IL-33', 'IL-35', 'OSM', 'IL-34', 'HGF', 'IL-1Ra', 'LIGHT', 'CD27L', 'SCF', 'IL-1-alpha', 'TGF-beta1', 'CD30L', 'IL-24', 'IFN-gamma', 'IF

In [5]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
%cd ~/multi-modal/Proposal2
from huggingface_hub import snapshot_download
snapshot_download("arcinstitute/ST-SE-Parse", local_dir="ST-SE-Parse",
                  allow_patterns=["fewshot/split_0/eval_best.ckpt/*"])
!ls -lh ST-SE-Parse/fewshot/split_0/eval_best.ckpt/

import anndata as ad
for name in ["adata_real", "adata_pred"]:
    a = ad.read_h5ad(f"ST-SE-Parse/fewshot/split_0/eval_best.ckpt/{name}.h5ad", backed="r")
    print(f"\n===== {name}: {a.shape}")
    print("obs columns:", a.obs.columns.tolist())
    for col in a.obs.columns:
        if a.obs[col].nunique() < 100:
            print(f"  {col}: {a.obs[col].nunique()} values, e.g. {list(a.obs[col].unique()[:6])}")
    print("var names:", list(a.var_names[:5]), "| obsm:", list(a.obsm.keys()))
    x = a.X[:5]; x = x.toarray() if hasattr(x, "toarray") else x
    print("X sample max:", float(x.max()), "| integer:", bool((x == x.round()).all()))

/home/w202gxg/multi-modal/Proposal2
total 6.9G
-rw-r--r-- 1 w202gxg 100000 3.6G Sep 28 06:25 adata_pred.h5ad
-rw-r--r-- 1 w202gxg 100000 3.6G Sep 28 06:25 adata_real.h5ad
-rw-r--r-- 1 w202gxg 100000 3.5K Sep 28 06:24 B_Intermediate_Memory_agg_results.csv
-rw-r--r-- 1 w202gxg 100000  15M Sep 28 06:24 B_Intermediate_Memory_pred_de.csv
-rw-r--r-- 1 w202gxg 100000  15M Sep 28 06:24 B_Intermediate_Memory_real_de.csv
-rw-r--r-- 1 w202gxg 100000  23K Sep 28 06:24 B_Intermediate_Memory_results.csv

===== adata_real: (235603, 2000)
obs columns: ['cytokine', 'cell_type_clean', 'donor', 'pert_cell_barcode', 'ctrl_cell_barcode']
  cytokine: 63 values, e.g. ['4-1BBL', 'IL-15', 'C5a', 'EGF', 'FGF-beta', 'CD30L']
  cell_type_clean: 1 values, e.g. ['B_Intermediate_Memory']
  donor: 12 values, e.g. ['Donor9', 'Donor1', 'Donor11', 'Donor12', 'Donor2', 'Donor4']
var names: ['0', '1', '2', '3', '4'] | obsm: ['X_state']
X sample max: 3.3258094787597656 | integer: False

===== adata_pred: (235603, 2000)
obs